# [sk_console_with_native_functions](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)

In just a few steps, you can build your first AI agent with Semantic Kernel in either Python, .NET, or Java. This guide will show you how to...

Install the necessary packages
Create a back-and-forth conversation with an AI
Give an AI agent the ability to run your code
Watch the AI create plans on the fly

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")
print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Initialize the kernel

In [2]:
from semantic_kernel import Kernel
kernel = Kernel()

# Add enterprise services (logging)

In [3]:
import logging

# Set the logging level for semantic_kernel.kernel to DEBUG.
logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create an Azure OpenAI chat completion, and add it to the Kernel

In [4]:
# Remove all services so that this cell can be re-run without restarting the kernel
kernel.remove_all_services()

from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
chat_completion=AzureChatCompletion(service_id="default")
kernel.add_service(chat_completion)

# Create a ***native*** plugin through a custom function

In [5]:
from typing import Annotated
from semantic_kernel.functions import kernel_function

class LightsPlugin:
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": True},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Testing the plugin (optional)

In [6]:
# Cell to execute each time you want to reset and toggle light 1
lp = LightsPlugin()  # (Re-)initialize lp to default state
lp.change_state(id=1, is_on=not(lp.lights[0]["is_on"]))  # Toggle light 1
print(lp.get_state())  # Print current state of all lights

[{'id': 0, 'name': 'Table Lamp', 'is_on': False}, {'id': 1, 'name': 'Porch light', 'is_on': False}, {'id': 2, 'name': 'Chandelier', 'is_on': True}]


# Add the native plugin to the Kernel

In [7]:
kernel.add_plugin(
    LightsPlugin(),
    plugin_name="Lights",
)

KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='the output is a string', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': 'the output is a string'}, include_in_function_choices=True), additional_properties={}), invocation_duratio

# Enable planning

In [8]:
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)

execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto() # Auto() or NoneInvoke()

# Chat history

In [9]:
from semantic_kernel.contents.chat_history import ChatHistory
# Create a history of the conversation
history = ChatHistory()
# Add user input to the history
history.add_user_message("Please toggle all the lights")

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Please toggle all the lights', encoding=None)], encoding=None, finish_reason=None)])

# Get the response from the AI

In [10]:
result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

#  Print the results
print("Assistant > " + str(result))

Assistant > [ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-Am2RlAivV8wGOYLfMDJI5mJrP3D3X', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='All the lights have been toggled successfully. The current states are:\n\n1. **Table Lamp:** Off\n2. **Porch Light:** Off\n3. **Chandelier:** On', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1736012513, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_04751d0b65', usage=CompletionUsage(completion_tokens=39, prompt_tokens=308, total_tokens=347, completion_tokens_details=None, prompt_tokens_details=None), prompt_filter_results=[{'prompt_index': 0, 'content_filter_r